# Dev Environment Setup

Setting up a development environment involves two core problems: (1) giving *ourselves* access to the machine and (2) giving the *machine* access to external services (GitHub, LLM APIs, etc). We focus here on remote environments using [Runpod](https://www.runpod.io/) GPU pods, but the same principles apply to any remote server (AWS EC2, GCP, Azure VMs) or even a fresh local setup.

SSH implements public-key cryptography to establish trust between client and server systems. In this paradigm, we have the **public key** (`.pub`) that functions as an access *verifier*, installed on target systems (like Runpod nodes), and the **private key** serves as the unique authentication factor for mathematically proving identity.

Runpod automatically loads the public keys to the pods making SSH access possible (@fig-runpod-ssh). Next, we give the pod read/write access to remote repositories in GitHub via a [personal access token](https://docs.github.com/en/authentication/keeping-your-account-and-data-secure/managing-your-personal-access-tokens) (PAT) which we store securely in Runpod **secrets manager**. Another common use-case of secrets are for storing API keys for LLM inference providers which can later be loaded into pods as **environmental variables**.

![**Setting up pods for development.** Mainly involves giving (1) *us* access to pods, and (2) *pods* access to external services (e.g. GitHub, OpenAI, etc). Not shown here is configuring the development environment (virtual envs, installing required libraries) once we have the initial code + further tooling (appendices).](./img/runpod-ssh.png){#fig-runpod-ssh}

:::{.callout-warning}
## Docker development limitation
Runpod is architecturally incompatible with standard Docker workflows (i.e. pods are themselves containers and cannot host a Docker daemon). For any development requiring local container orchestration or image building, use a traditional cloud VM (e.g. AWS EC2, GCP Compute Engine) where you have full control over the host machine.

:::

## Give local SSH access to pods

Generate SSH keys for Runpod (no passphrase):
```{.bash filename="$ (local)"}
ssh-keygen -t ed25519 -C "runpod" -f ~/.ssh/id_ed25519_runpod -N ""
cat ~/.ssh/id_ed25519_runpod.pub
```
Append the output to **Settings**> **SSH Public Keys** in Runpod (separated by newline).

## Give pods SSH access to GitHub

Go to **Settings>Developer Settings** in GitHub to generate a fine-grained **personal access token** (PAT).
Make sure to apply the appropriate repository permissions. It is also good practice to only allow access to select 
repositories: 

![](./img/runpod-github-pat.png)

Add the generated token to Runpod **secrets manager**:

![](./img/runpod-secrets-manager.png)


## Pod creation

During pod creation, load secrets (e.g. the GitHub PAT) as **environmental variable**. Note that we reduce the container disk
to the minimum required value since we typically work on the **pod volume** mounted on `/workspace` which persists data 
even when the pod is stopped[^volumecost]. Also select the appropriate image (here we choose the image with PyTorch 2.8 support):

![](./img/runpod-pod-creation.png)

This is also the place to load other API keys.
Once the env vars are added,
connect using **SSH over exposed TCP** [connection string]{.underline}[^connstring]. Inside the pod you can check
that the access tokens have been loaded using:
```{.bash filename="$ (pod)"}
env | grep GITHUB
```

Then, we can clone [our repository](https://github.com/particle1331/ai-notebooks) via HTTPS with the required access using:
```{.bash filename="$ (pod)"}
cd /workspace
export USERNAME=particle1331
export GITHUB_REPO=github.com/particle1331/ai-notebooks
git clone https://${USERNAME}:${GITHUB_TOKEN}@${GITHUB_REPO}.git
cd ai-notebooks
```

:::{.callout-tip}
You can try `git push` to test for write access as this would fail if the PAT was improperly loaded.

:::

[^connstring]: e.g. `ssh root@63.141.33.33 -p 22011 -i ~/.ssh/id_ed25519_runpod`

[^volumecost]: A stopped pod still has an hourly cost (e.g. 0.01$ / hr) so it's a good idea to terminate a pod before a long break. Note that this means losing the data in `/workspace`.

## Virtual environment

Our recommended approach is to use `uv` to build a venv at `.venv` that is synced using `uv.lock`. This environment can then be used as Jupyter kernel [in vscode](https://code.visualstudio.com/docs/datascience/jupyter-kernel-management). Or run scripts using [`uv run`](https://docs.astral.sh/uv/concepts/projects/run/#running-scripts). The required Python version is also specified ([-@lst-Makefile]):

```{.bash filename="$ (pod)"}
make venv
```

:::{.callout-tip}
Some environments can cause `uv` failures (e.g. servers with network-mounted filesystems like AzureML compute instances). 
In this situation, you can use `pip` to install packages without using `uv` as dependency manager:
```bash
# install required python version, pip, and activate venv
make uv
uv python install 3.13
uv venv .venv && source .venv/bin/activate
curl -sS https://bootstrap.pypa.io/get-pip.py | .venv/bin/python

# install requirements on venv
make requirements
uv pip install -r requirements.txt
uv pip install -e .
```

:::

:::{.callout-note}
In the case of short-lived pods, you don't necessarily need to perfectly setup an environment or maintain a clean state. However, reproducibility benefits from a controlled, well-defined environment. As usual, the level of precision needed depends on the scope of the project.

:::

## Appendix: Quarto docs

Check [here](https://github.com/particle1331/ai-notebooks/blob/main/.github/workflows/publish.yml) for the Quatro version that we're using. Adjust the following variable accordingly:
```{.bash filename="$ (pod)"}
export QUARTO_VERSION=1.7.32  # may be outdated
wget -q https://github.com/quarto-dev/quarto-cli/releases/download/v${QUARTO_VERSION}/quarto-${QUARTO_VERSION}-linux-amd64.deb
dpkg -i quarto-${QUARTO_VERSION}-linux-amd64.deb && rm quarto-${QUARTO_VERSION}-linux-amd64.deb
```

Preview then [port-forward](https://code.visualstudio.com/docs/remote/ssh#_forwarding-a-port-creating-ssh-tunnel) using vscode:
```{.bash filename="$ (pod)"}
make docs
```

## Appendix: Tmux

Tmux allows you to open multiple windows in a **single SSH session** to a remote server, without needing to authenticate separately for each window. You can also detach from a tmux session and reconnect later to resume exactly where you left off. This means you can run several tasks in parallel, switch between them easily, and keep them running even if your connection drops. ◧

For convenience, I use [byobu](https://www.byobu.org/). First, [install byobu](https://www.byobu.org/downloads) in the remote server. This will run a **tmux daemon** on the server which persists the session. In the following table, I list commands and workflows I found useful:

```{.bash filename="$ (pod)"}
apt-get update
apt-get install -y byobu
byobu
```

| Command | Function |
| :--: | :-- |
| F1, Shift + F1 | Display help |
| Ctrl + F2 | Create new split vertically |
| Shift + F2 | Create new split horizontally |
| F2 | Create new window |
| F8 | Rename the current window |
| Ctrl + F6 | Kill a focused split |
| F3 / F4 | Switch between windows |
| Shift + F3 / F4 | Switch between splits |
| F6 | Detach session |
| Shift + F9 | Run command on all splits. See @fig-byobu-command-splits. |
: Byobu commands {tbl-colwidths="[30,70]"}

:::{.callout-caution}
Make sure to disable your system keyboard shortcuts for these combinations! For example, (ꐦ¬_¬) Apple has a default shortcut for ⌃F2, which translates to Ctrl+F2, resulting in some head-scratching if you're unaware! (It similarly has shortcuts for ⌃F3, ⌃F6, etc.) 

:::

Running the same command on multiple splits (very useful):

![Running a dynamic command on 3 splits.](./img/byobu-commands.png){#fig-byobu-command-splits}

**Resuming.** Working on a remote server, you might lose your connection unexpectedly or intentionally disconnect while wanting to keep processes running. In such cases, you can **press F6** in Byobu to detach the session and logout. Once you SSH again to the server, the session will be restored by running `byobu`. This is very useful!

![Resuming a detached session. The process kept running in the background while we were away.](./img/byobu-detached.png){#fig-byobu-detached}

## Appendix: Pi coding agent {#sec-pi-agent}

[Pi](https://pi.dev/) is an AI coding agent that runs in the terminal (TUI) or as a GUI app. It supports multiple LLM providers, tool use, extensions, and project-specific configuration via a `.pi/` directory at the project root. Installing it on a new machine (remote or local) involves two steps: (1) installing pi itself and (2) syncing extensions from the project's configuration.

**Installing pi.** On Linux (e.g. inside a Runpod pod), install pi using npm:
```{.bash filename="$ (pod)"}
npm install -g --ignore-scripts @earendil-works/pi-coding-agent
```

**Extensions.** Pi extensions are npm packages that add functionality (themes, tools, status bars, subagents, etc). The project's extension configuration lives in `.pi/settings.json`.

The `packages` array lists all extensions the project uses. When setting up a new environment, pi reads this file and installs the listed packages automatically on first launch. The actual npm dependencies are resolved in `.pi/npm/package.json` (gitignored along with `node_modules/`). To manually trigger extension installation:

```{.bash filename="$ (pod)"}
cd /workspace/ai-notebooks/.pi/npm
npm install
```



**Project structure.** Beyond extensions, the `.pi/` directory can contain:

| Path | Purpose |
| :-- | :-- |
| `.pi/settings.json` | Extension packages, subagent config, model overrides |
| `.pi/agents/` | Custom agent definitions (specialized roles with tool access) |
| `.pi/skills/` | Reusable instruction sets loaded on demand |
| `.pi/prompts/` | Prompt templates (triggered via `/` in the TUI) |

<br>

### Skills

A [skill](https://pi.dev/docs/latest/skills) is a self-contained capability 
package that the agent loads on demand. Each skill is a directory containing a `SKILL.md` 
file with YAML frontmatter (`name`, `description`) and detailed instructions. The mechanism is 
**progressive disclosure**: at startup, pi includes only skill names and descriptions in the 
system prompt[^skill_prompt]. The full `SKILL.md` content is loaded later via 
the `read` tool when a task matches, or forced explicitly with `/skill:name`.

[^skill_prompt]: via `<available_skills>` XML.

A skill directory can also contain scripts, reference docs, and assets:

```bash
.pi/skills/matplotlib-style/
├── SKILL.md          # frontmatter + instructions
├── references/       # detailed docs loaded on-demand
└── assets/           # templates, configs, etc.
```


The `SKILL.md` frontmatter declares when the skill should be used:

```{.yaml filename=".pi/skills/matplotlib-style/SKILL.md"}
---
name: matplotlib-style
description: >
  Matplotlib and seaborn plotting conventions extracted from 134
  plotting cells across the ai-notebooks project. Covers figure
  creation, color palettes, axis styling, annotations, multi-panel
  layouts, and Quarto integration. Load this skill when creating
  or reviewing any visualization code.
---
```

The rest of the document contains usual Markdown-structured contents such as code snippets, coding standards, and conventions.
A skill can be generated using coding assistants. For example, one can provide existing
codebases, curated docs, or repositories that contain the scoped / specific practices that can be distilled by an agent
into the skill. In fact, there is a `skill-creator` skill from the [anthropic/skills](https://github.com/anthropics/skills) library exactly for this purpose.

:::{.callout-note}
There are four custom skills specific to this project: `flet-dev` (Flet API reference), 
`matplotlib-style` (plotting conventions), `notebook-writing-style` (prose style guide), and `quarto-dev` (Quarto syntax). These encode project-specific conventions that would be too verbose to include in every prompt but are critical for consistency.
These are distilled from manually written curated content. Some generic skills that we also use are `xlsx` (for handling excel) and `pptx` (creating powerpoints) are downloaded from the [anthropic/skills](https://github.com/anthropics/skills) library.
:::

### Prompts

Files in `.pi/prompts/` are reusable [prompt templates](https://pi.dev/docs/latest/prompt-templates) 
invoked by typing `/name` in the TUI, where `name` is the filename without `.md`. 
Templates are markdown snippets that expand into full prompts with support for positional arguments 
for filling in values in the template:

| Syntax | Meaning |
| :-- | :-- |
| `$1`, `$2`, ... | Positional arguments |
| `$@` or `$ARGUMENTS` | All arguments joined |
| `${@:N}` | Arguments from position N onward |
| `${@:N:L}` | L arguments starting at position N |
: Prompt template argument syntax {tbl-colwidths="[30,70]"}

Each template has optional YAML frontmatter with `description` 
(shown in autocomplete) and `argument-hint` (shows expected arguments, e.g. 
`<notebook-path> [difficulty]` &mdash; a required argument followed by an optional one, 
with default values are simply stated in the prompt as text). For example, this project has an `exercise-gen.md` 
template that converts an explanatory notebook into a scaffolded exercise version:

```bash
/exercise-gen notebooks/deep/01-softmax-regression.ipynb hard true
```

<details>
<summary>.pi/prompts/exercise-gen.md</summary>

````md
---
description: Generate exercise notebook from explanatory notebook
argument-hint: "<notebook-path> [difficulty] [hints]"
---

I need you to generate an exercise notebook from an explanatory notebook. Use the exercise generation pipeline.

**Input notebook:** $1 (the notebook path to convert)

**Output location:** `tmp/EXERCISE_$1` (preserve the original filename with EXERCISE_ prefix)

**Task:**
1. Load the notebook from the given path
2. [...]
7. Write the output notebook to `tmp/EXERCISE_<original_filename>.ipynb`

**Difficulty level:** $2 (default: medium)
- easy: leave variable names and helper call names as hints
- medium: remove helper call names but keep structure
- hard: minimal scaffolding, only signature and docstring

**Include hints:** $3 (default: false)
- If true, insert a collapsible hint cell after each exercise cell using HTML `<details>`/`<summary>` tags, which VS Code renders as a clickable disclosure widget in markdown cells:

```
<details>
<summary>Hint</summary>

```text
<pseudocode>
```

</details>
```
[...]
````
</details>

Prompt templates usage mimics usual CLI commands although what's happening is just building prompts for the agent 
to ingest. The agent would need tools to execute the contents of the prompt. Moreover, unlike a true CLI
command there is no formal guarantee that the LLM will follow the command and its parameters. 

:::{.callout-tip}
**Global prompts.** Pi also loads templates from global (`~/.pi/agent/prompts/`) and from installed packages, so common workflows can be shared across projects. This makes it easy to standardize complex multi-step workflows into a single invocation.

**Git.** The entire `.pi/` configuration is version-controlled alongside the code. Cloning the repository and launching pi is sufficient to reproduce the full development environment, including specialized agents, skills, and prompt templates.

:::

### Subagents

A [subagent](https://pi.dev/packages/pi-subagents) is an extension to pi that allows a [specialized agent]{.underline} spawned as an **isolated subprocess** to handle a delegated task. Each subagent gets its own context window, system prompt, tool set, and (optionally) a different model. Agent definitions live in `.pi/agents/` as markdown files with YAML frontmatter. Here we define the custom subagent:

```{.yaml filename=".pi/agents/notebook-writer.md"}
---
name: notebook-writer
description: >
  Writes and reviews Jupyter notebook content matching the author's distinctive
  pedagogical style. Use this agent when creating new notebook cells, reviewing
  notebook prose quality, or rewriting sections to match the established voice
  and formatting conventions. Specialized for ai-notebooks project.
model: claude-sonnet-4.6
thinking: medium
tools: read, edit, write
inheritProjectContext: true
skills: quarto-dev, matplotlib-style
---

You are a notebook writing assistant for the **ai-notebooks** project. Your job is to
write, review, and improve Jupyter notebook content that precisely matches the author's
established style. The author is highly critical of their own work, so quality and
consistency matter enormously.

[...]
```

:::{.callout-tip}
See [docs](https://pi.dev/packages/pi-subagents) for off-the-shelf subagents such as `scout`, `researcher`, `oracle`, and `planner` that are provided by the extension.

:::

**Why not just use the main agent?** Four reasons:

1. **Isolated context windows.** The main agent's context fills up as a conversation progresses. A subagent starts fresh, so it can focus entirely on the delegated task without being distracted by prior conversation history. When it finishes, only its *output* is returned to the parent — not its entire reasoning trace.

2. **Specialization and cost control.** Different tasks have different requirements. A quick codebase scan can use a fast, cheap model (Haiku) with read-only tools, while a complex refactor needs a powerful model (Sonnet/Opus) with write access. Subagents let you match the model and toolset to the task granularity.

3. **Parallel execution.** Multiple subagents can run concurrently. For example, scouting different parts of a codebase simultaneously — which is impossible with a single serial agent.

4. **Tools restriction.** Pi doesn't support plan mode by default. But we circumvent this by having a planner subagent. However, to ensure that the planner does not edit the files, we only give it read tools such as `read`, `web_search`, and `web_fetch`. Allowed tools is set in the `tools:` field of the subagent YAML.

The subagent tool supports three execution modes: **single** (one agent, one task), **parallel** (multiple agents running concurrently), and **chain** (sequential handoff where each agent receives the previous agent's output). A typical workflow chains a scout (cheap, fast reconnaissance) into a planner (structured implementation plan) into a worker (full-capability implementation). Refer to the [official docs](https://pi.dev/packages/pi-subagents) for the syntax and details about the default subagents (`scout`, `oracle`, etc. -- see @fig-pi-subagents). Some useful commands:

```bash
/chain scout "scan the codebase" -> planner "create an implementation plan"   # <1> 
/chain scout planner -- analyze the auth system   # <2> 
/parallel scanner "find security issues" -> reviewer "check code style" # <3>
/run worker[model=claude-opus-4.6,skills=flet-dev+matplotlib-style,output=plot_sine.py] Plot a sine wave in UI # <4>
```
1. Sequential tasks with handoff.
2. Use `--` for a shared task.
3. Running independent parallel tasks. Here `->` syntax acts as separator.
4. Running with a specific model, skills, and output path. You can also specify `reads` to give the agent a file to read. Here the model uses Flet and matplotlib without being explicitly told to. See [output below](@fig-flet-sine).

:::{.callout-tip}
The `pi-subagents` extension already provides a [corresponding skill](https://github.com/nicobailon/pi-subagents/blob/main/skills/pi-subagents/SKILL.md). So you can expect the main orchestrator in the parent session to have the ability to run subagents in various execution modes. Hence, you don't need to run the above commands unless you want to be really intentional. Refer to the ff exchange which demonstrates the skill:

![Asking pi to write two stories in parallel.](./img/subagents-chat.png){#fig-pi-subagents}

:::

<details>
<summary>TUI rendering of the worker subagent's execution:</summary>

````bash
ok worker [fork] | 20 tools, 4.4k tok, 2m31s

 Task: You are a delegated subagent running from a fork of the parent session. Treat the
 inherited conversation as reference-only context, not a live thread to continue. Do not continue
 or answer prior messages as if they are waiting for a reply. Your sole job is to execute the
 task below and return a focused result for that task using your tools.

 Task:
 Plot a sine wave in UI

 ---
 **Output:** Write your findings to: /Users/particle1331/code/latest/ai-notebooks/plot_sine.py

 $ cd /Users/particle1331/code/latest/ai-notebooks && python3 -c "import flet;
 print(flet.__version__)"
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "import flet;
 print(flet.__version__)" 2>/dev/null || echo "no flet"
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "import flet as ft;
 print(dir(ft))" | tr ',' '\n' | grep -i "image\|chart\|plot\|canvas"
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "import flet as ft;
 print([x for x in dir(ft) if 'chart' in x.lower() or 'line' in x.lower()])"
 write ~/code/latest/ai-notebooks/plot_sine.py
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 import sys, importlib
 # Verify all imports work
 for mod in ['flet', 'matplotlib.pyplot', 'numpy', 'base64', 'io']:
     importlib.import_module(mod)
     print(f'  ✓ {mo...
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 # Dry-run: test that make_sine_plot produces valid base64 PNG
 import sys
 sys.path.insert(0, '.')
 from plot_sine import make_sine_plot
 b64 = make_sine_plot()
 import bas...
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "import flet as ft;
 print(hasattr(ft, 'run'), hasattr(ft, 'app'))"
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 import inspect, flet as ft
 sig = inspect.signature(ft.app)
 print('ft.app params:', list(sig.parameters.keys()))
 "
 $ cd /Users/particle1331/code/latest/ai-notebooks && grep -r "ft.app\|ft.run"
 notebooks/apps/01-flet.ipynb 2>/dev/null | head -5
 edit ~/code/latest/ai-notebooks/plot_sine.py
 read ~/code/latest/ai-notebooks/plot_sine.py
 edit ~/code/latest/ai-notebooks/plot_sine.py
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 import flet as ft
 import inspect

 # Check Image constructor
 sig = inspect.signature(ft.Image.__init__)
 print('ft.Image params:')
 for name, param in sig.parameters.item...
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 import flet as ft
 # Check if src accepts base64
 help_text = ft.Image.__doc__ or ''
 print('Image doc:', help_text[:500])
 print()
 # Check ImageFit
 print('ImageFit:', dir...
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 import flet as ft
 # Check what fit-related things exist
 print([x for x in dir(ft) if 'fit' in x.lower()])
 # Check if src can take base64
 img = ft.Image(src='test')
 pri...
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 import flet as ft
 # In Flet, src can accept base64 with a data URI prefix
 # Check BoxFit
 print('BoxFit:', [x for x in dir(ft.BoxFit) if not x.startswith('_')])
 "
 edit ~/code/latest/ai-notebooks/plot_sine.py
 read ~/code/latest/ai-notebooks/plot_sine.py
 $ cd /Users/particle1331/code/latest/ai-notebooks && .venv/bin/python -c "
 # Validate script compiles and make_sine_plot runs without launching UI
 import sys, importlib.util
 spec = importlib.util.spec_from_file_location('plot_sine', 'plot_sin...

 """Flet app that plots a sine wave using matplotlib rendered to an in-memory image."""

 import base64
 import io

 import flet as ft
 import matplotlib.pyplot as plt
 import numpy as np

 def make_sine_plot() -> str:
     """Render a sine wave with matplotlib and return a base64-encoded PNG."""
     x = np.linspace(0, 4 * np.pi, 500)
     y = np.sin(x)

 ```
   fig, ax = plt.subplots(figsize=(8, 3))
   ax.plot(x, y, color="C0", linewidth=2)
   ax.set_xlabel("x")
   ax.set_ylabel("sin(x)")
   ax.set_title("Sine Wave")
   ax.grid(linestyle="dotted", alpha=0.6)
   fig.tight_layout()

   buf = io.BytesIO()
   fig.savefig(buf, format="png", dpi=150)
   plt.close(fig)
   buf.seek(0)
   return base64.b64encode(buf.read()).decode("utf-8")
 ```

 def main(page: ft.Page):
     page.title = "Sine Wave Plot"
     page.vertical_alignment = ft.MainAxisAlignment.CENTER
     page.horizontal_alignment = ft.CrossAxisAlignment.CENTER

 ```
   img_b64 = make_sine_plot()

   page.add(
       ft.Image(
           src=f"data:image/png;base64,{img_b64}",
           width=800,
           fit=ft.BoxFit.CONTAIN,
       )
   )
 ```

 ft.run(main)

 Skills: flet-dev, matplotlib-style
 21 turns in:23 out:4.4k R732k W78k github-copilot/claude-opus-4.6:high
````
</details>

![Worker subagent outputs runnable Python file which opens a Flet UI of a sine wave.](./img/flet-sine.png){#fig-flet-sine}

## Appendix: Monorepos with uv {#sec-uv-workspaces}

[uv](https://docs.astral.sh/uv/) is a fast Python package manager that supports **workspaces** for managing monorepos containing multiple related packages. A workspace is defined by a root `pyproject.toml` that declares member projects, each with their own `pyproject.toml` and dependency specifications. All members share a single `uv.lock` file and a single `.venv`.

**Project layout.** Workspace members are typically placed in a `projects/` or `packages/` directory:

```bash
ai-notebooks/
├── pyproject.toml          # root project + workspace declaration
├── uv.lock                 # single lockfile for all members
├── .venv/                  # shared virtual environment
├── src/notebooks/          # root project source
└── projects/
    └── flet-basics/
        ├── pyproject.toml  # member project
        └── src/flet_basics/
```


**Workspace configuration.** In the root `pyproject.toml`, declare workspace members:

```{.toml filename="pyproject.toml (root)"}
[project]
name = "ai-notebooks"
requires-python = ">=3.13"
dependencies = [
    "torch>=2.7.1",
    "numpy>=2.3.1",
    # ... main project deps
]

[tool.uv.workspace]
members = ["projects/flet-basics"]
```

Each member has its own `pyproject.toml`:

```{.toml filename="projects/flet-basics/pyproject.toml"}
[project]
name = "flet-basics"
requires-python = ">=3.9"
dependencies = [
    "flet>=0.80.1",
    "redis>=7.1.0",
]
```

Each workspace member declares only the dependencies it directly needs. The shared `.venv` installs the *union* of all member dependencies — but each `pyproject.toml` stays scoped to its own project. The root does not accumulate subproject deps (e.g. `flet`, `redis`), and a new member does not inherit the root's packages automatically. This separation keeps every manifest auditable: inspecting any `pyproject.toml` gives an accurate picture of what that project requires.

**Basic workflows:**

```{.bash filename="$ (local)"}
# initialize a new member project — uv auto-adds it to [tool.uv.workspace].members
uv init projects/flet-basics

# sync all workspace members into the shared .venv
uv sync

# add a dependency to the root project
uv add requests

# add a dependency to a specific member
uv add --package flet-basics some-library

# run a script in the context of a specific member
uv run --package flet-basics python -m flet_basics.main

# lock without installing (useful in CI)
uv lock
```


:::{.callout-important}
**Conflicting dependency error.** A uv workspace uses a [single unified resolution](https://docs.astral.sh/uv/concepts/workspaces/#dependency-resolution) across all members. This means [all workspace members share the same version]{.mark} of any given package. If the root project requires `flet>=0.80.1` and a member also requires `flet>=0.80.1`, uv resolves to a single version that satisfies both constraints. This design is intentional: it guarantees consistency across the monorepo and avoids subtle bugs from version mismatches. For example, if one member pins `numpy<2.0` while another requires `numpy>=2.3`, `uv lock` will fail with an unsatisfiable dependency error. 

**Root Python version.** The lockfile respects `requires-python` bounds, which can influence which package versions are eligible. In our case, the root requires `>=3.13` while `flet-basics` declares `>=3.9`, but since the workspace `.venv` uses the root's Python, the effective resolution uses 3.13+.

:::


:::{.callout-tip}
Running `uv sync` from the repository root installs all members as [editable packages]{.mark} into the shared `.venv`. This means [imports work across the entire workspace]{.underline} without any path manipulation!

:::

## Appendix: Code listings

---

■